# LeetCode 1979 — Find Greatest Common Divisor of Array bằng Python

**Link đề:** https://leetcode.com/problems/find-greatest-common-divisor-of-array/
**Độ khó:** Easy · **Chủ đề:** Array, Math, Number Theory, Euclidean Algorithm

```python
class Solution:
    def findGCD(self, nums: List[int]) -> int:

```

## Mục tiêu học
1. Đọc đúng đề: **không** tính GCD của cả mảng, mà chỉ GCD của **số nhỏ nhất** và **số lớn nhất**.
2. Nắm các công cụ Python: `min`, `max`, toán tử `%` (chia lấy dư), vòng `while`, gán đồng thời `a, b = b, a % b`, `math.gcd`.
3. Viết cách trực giác: thử từng ước từ `mn` xuống `1`.
4. Hiểu **thuật toán Euclid** — vì sao `gcd(a, b) = gcd(b, a % b)` — và trace từng bước.
5. So sánh 4 cách (thử ước, Euclid lặp, Euclid đệ quy, `math.gcd`), viết test đối chiếu, làm bài tập biến thể.

## 1. Đề bài yêu cầu gì?

Cho mảng số nguyên `nums`. Hãy trả về **ước chung lớn nhất (GCD — Greatest Common Divisor)** của **số nhỏ nhất** và **số lớn nhất** trong `nums`.

> GCD của hai số là **số nguyên dương lớn nhất** chia hết cả hai số.

| `nums` | min | max | Kết quả | Vì sao? |
|---|:-:|:-:|:-:|---|
| `[2,5,6,9,10]` | 2 | 10 | `2` | 2 chia hết 2 và 10 |
| `[7,5,6,8,3]` | 3 | 8 | `1` | 3 và 8 không có ước chung nào lớn hơn 1 |
| `[3,3]` | 3 | 3 | `3` | GCD của một số với chính nó là chính nó |

**Constraints:**
- `2 <= nums.length <= 1000`
- `1 <= nums[i] <= 1000` → mọi số đều **dương**, không có số 0, không có số âm.

Bài có 2 bước tách bạch: **(1)** tìm `mn`, `mx`; **(2)** tính `gcd(mn, mx)`.

In [1]:
# Chạy thử tay phần "đọc đề": chỉ quan tâm min và max
nums = [2, 5, 6, 9, 10]
print("nums =", nums)
print("nhỏ nhất:", min(nums))
print("lớn nhất:", max(nums))

nums = [2, 5, 6, 9, 10]
nhỏ nhất: 2
lớn nhất: 10


## 2. Nền tảng Python cần biết

### 2.1. `min()` và `max()` trên list

- `min(nums)` duyệt toàn bộ list và trả về phần tử nhỏ nhất.
- `max(nums)` trả về phần tử lớn nhất.
- Mỗi hàm duyệt list **một lần**, tốn `O(n)`.

In [2]:
print(min([7, 5, 6, 8, 3]))   # 3
print(max([7, 5, 6, 8, 3]))   # 8
print(min([3, 3]), max([3, 3]))  # 3 3 — min và max có thể bằng nhau

3
8
3 3


### 2.2. Tự tìm min và max trong **một** vòng lặp

Hint của đề gợi ý: tìm cả hai trong một lần duyệt. Ta giữ hai biến `mn`, `mx`, khởi tạo bằng phần tử đầu tiên, rồi cập nhật khi gặp số nhỏ hơn / lớn hơn.

Vì sao khởi tạo bằng `nums[0]` mà không phải `0`? Vì nếu `mn = 0` thì không số dương nào nhỏ hơn 0 → `mn` sai mãi.

In [3]:
def min_max_one_pass(nums):
    mn = mx = nums[0]          # gán cùng một giá trị cho 2 biến
    for x in nums[1:]:         # nums[1:] = list bỏ phần tử đầu
        if x < mn:
            mn = x
        if x > mx:
            mx = x
    return mn, mx              # trả về tuple 2 phần tử

print(min_max_one_pass([2, 5, 6, 9, 10]))
print(min_max_one_pass([7, 5, 6, 8, 3]))

(2, 10)
(3, 8)


### 2.3. Toán tử `%` — chia lấy dư

`a % b` là **phần dư** khi chia `a` cho `b`.

- `10 % 2 == 0` → 2 **chia hết** 10 (2 là ước của 10).
- `10 % 4 == 2` → 4 không chia hết 10.

Đây là cách kiểm tra "`d` có phải ước của `a`?" : `a % d == 0`.

In [4]:
for a, d in [(10, 2), (10, 4), (8, 3), (9, 3)]:
    print(f"{a} % {d} = {a % d}  ->  {d} {'là' if a % d == 0 else 'không là'} ước của {a}")

10 % 2 = 0  ->  2 là ước của 10
10 % 4 = 2  ->  4 không là ước của 10
8 % 3 = 2  ->  3 không là ước của 8
9 % 3 = 0  ->  3 là ước của 9


### 2.4. `range` đi lùi

`range(start, stop, step)` với `step = -1` đi lùi. `stop` **không** được lấy.

`range(5, 0, -1)` → `5, 4, 3, 2, 1` (dừng trước 0).

In [5]:
print(list(range(5, 0, -1)))
print(list(range(3, 0, -1)))

[5, 4, 3, 2, 1]
[3, 2, 1]


### 2.5. Vòng `while` và gán đồng thời `a, b = b, a % b`

Python cho phép gán nhiều biến cùng lúc. Vế phải được **tính xong hết trước**, rồi mới gán vào vế trái.

```python
a, b = b, a % b
```

nghĩa là: `a_mới = b_cũ`, `b_mới = a_cũ % b_cũ`. Nếu viết hai dòng riêng thì sai vì `a` đã bị ghi đè trước khi tính `a % b`.

In [6]:
a, b = 10, 4
a, b = b, a % b       # dùng a=10, b=4 cũ để tính
print("gán đồng thời:", a, b)   # 4 2

a, b = 10, 4
a = b                 # a = 4
b = a % b             # 4 % 4 = 0  -> sai ý đồ!
print("gán tách dòng (sai):", a, b)

gán đồng thời: 4 2
gán tách dòng (sai): 4 0


### 2.6. `math.gcd` có sẵn trong thư viện chuẩn

Module `math` có hàm `gcd(a, b)`. Từ Python 3.9, `math.gcd` nhận **nhiều** đối số: `math.gcd(12, 18, 30)`.

Dùng được trên LeetCode, nhưng ta vẫn nên hiểu nó hoạt động thế nào (Euclid).

In [7]:
import math
print(math.gcd(2, 10))   # 2
print(math.gcd(3, 8))    # 1
print(math.gcd(3, 3))    # 3
print(math.gcd(12, 18, 30))  # 6 (Python >= 3.9)

2
1
3
6


## 3. Hiểu GCD bằng tay trước khi code

Liệt kê ước của từng số, rồi tìm **ước chung lớn nhất**:

| Số | Các ước |
|---|---|
| 12 | 1, 2, 3, 4, 6, 12 |
| 18 | 1, 2, 3, 6, 9, 18 |
| Ước chung | 1, 2, 3, **6** |

→ `gcd(12, 18) = 6`.

Nhận xét quan trọng: GCD **không thể lớn hơn số nhỏ hơn** trong hai số (vì ước của `mn` không vượt quá `mn`). Nên chỉ cần thử `d` trong `[1, mn]`.

In [8]:
def divisors(a):
    return [d for d in range(1, a + 1) if a % d == 0]   # list comprehension: lọc d chia hết a

print("ước 12:", divisors(12))
print("ước 18:", divisors(18))
common = [d for d in divisors(12) if 18 % d == 0]
print("ước chung:", common, "-> lớn nhất:", max(common))

ước 12: [1, 2, 3, 4, 6, 12]
ước 18: [1, 2, 3, 6, 9, 18]
ước chung: [1, 2, 3, 6] -> lớn nhất: 6


### Giải thích list comprehension

```python
[d for d in range(1, a + 1) if a % d == 0]
```

đọc là: "tạo list gồm các `d`, với `d` chạy từ 1 tới `a`, **chỉ giữ** những `d` mà `a % d == 0`". Tương đương vòng `for` + `if` + `append`, nhưng viết gọn một dòng.

## 4. Cách 1 — Trực giác: thử ước từ `mn` lùi về `1`

Ý tưởng (đúng như hint 2 của đề):
1. Tìm `mn`, `mx`.
2. Thử `d = mn, mn-1, ..., 1`. Ước **đầu tiên** chia hết cả `mn` và `mx` chính là **lớn nhất** (vì ta đi từ lớn xuống).
3. Vòng lặp chắc chắn dừng vì `d = 1` luôn chia hết mọi số.

In [9]:
def find_gcd_try_divisors(nums):
    mn, mx = min(nums), max(nums)
    for d in range(mn, 0, -1):
        if mn % d == 0 and mx % d == 0:
            return d
    return 1   # không bao giờ tới đây vì d=1 luôn thỏa

print(find_gcd_try_divisors([2, 5, 6, 9, 10]))  # 2
print(find_gcd_try_divisors([7, 5, 6, 8, 3]))   # 1
print(find_gcd_try_divisors([3, 3]))            # 3

2
1
3


### Code nộp LeetCode — Cách 1

In [10]:
from typing import List

class Solution:
    def findGCD(self, nums: List[int]) -> int:
        mn, mx = min(nums), max(nums)
        for d in range(mn, 0, -1):
            if mn % d == 0 and mx % d == 0:
                return d
        return 1

print(Solution().findGCD([2, 5, 6, 9, 10]))

2


### Độ phức tạp Cách 1

Gọi `n = len(nums)`, `V = mn` (giá trị nhỏ nhất).
- Tìm min/max: `O(n)`.
- Thử ước: tệ nhất `O(V)` lần (ví dụ `mn = 997`, `mx = 1000` nguyên tố cùng nhau → thử gần 997 số).
- Tổng: **Time `O(n + V)`**, **Space `O(1)`**.

Với constraints `V ≤ 1000` thì quá nhanh. Nhưng nếu giá trị lên tới `10^18`, thử từng số là không thể.

In [11]:
def count_tries(mn, mx):
    tries = 0
    for d in range(mn, 0, -1):
        tries += 1
        if mn % d == 0 and mx % d == 0:
            return d, tries

print("gcd, số lần thử (997, 1000):", count_tries(997, 1000))
print("gcd, số lần thử (500, 1000):", count_tries(500, 1000))

gcd, số lần thử (997, 1000): (1, 997)
gcd, số lần thử (500, 1000): (500, 1)


## 5. Nút thắt và insight — Thuật toán Euclid

**Nút thắt:** Cách 1 thử *mù* từng số. Số lần thử tỉ lệ với **giá trị** `mn`, không phải số chữ số.

**Insight (Euclid, ~300 TCN):** với `a ≥ b > 0`:

$$\gcd(a, b) = \gcd(b,\ a \bmod b), \qquad \gcd(a, 0) = a$$

**Vì sao?** Viết `a = q·b + r` (với `r = a % b`).
- Nếu `d` chia hết `a` và `b` thì `d` chia hết `r = a - q·b`.
- Ngược lại, nếu `d` chia hết `b` và `r` thì `d` chia hết `a = q·b + r`.

→ Tập **ước chung** của `(a, b)` và của `(b, r)` là **y hệt nhau**, nên ước chung lớn nhất cũng như nhau. Mỗi bước số nhỏ đi rất nhanh, đến khi `b = 0` thì đáp án là `a`.

In [12]:
# Kiểm chứng: tập ước chung của (a, b) và (b, a % b) trùng nhau
def common_divisors(a, b):
    return {d for d in range(1, max(a, b) + 1) if a % d == 0 and b % d == 0}   # set comprehension

a, b = 48, 18
print("ước chung (48, 18):", sorted(common_divisors(a, b)))
print("ước chung (18, 48 % 18 = 12):", sorted(common_divisors(b, a % b)))

ước chung (48, 18): [1, 2, 3, 6]
ước chung (18, 48 % 18 = 12): [1, 2, 3, 6]


### Ví dụ tay: `gcd(48, 18)`

| Bước | a | b | a % b | Ý nghĩa |
|:-:|:-:|:-:|:-:|---|
| 1 | 48 | 18 | 12 | 48 = 2·18 + 12 |
| 2 | 18 | 12 | 6 | 18 = 1·12 + 6 |
| 3 | 12 | 6 | 0 | 12 = 2·6 + 0 |
| 4 | 6 | 0 | — | `b = 0` → dừng, đáp án **6** |

Chỉ 3 phép chia! So với thử ước từ 18 xuống 6 (13 lần thử).

## 6. Cách 2 — Euclid dạng vòng lặp (lời giải chính)

Giảng từng đoạn:

```python
mn, mx = min(nums), max(nums)   # bước 1: chỉ cần 2 số này
a, b = mx, mn                   # bước 2: chuẩn bị cho Euclid
while b != 0:                   # bước 3: còn dư thì còn chia
    a, b = b, a % b             #         (a, b) -> (b, a % b)
return a                        # bước 4: b = 0 thì a là GCD
```

Ghi chú: thứ tự `a, b` không quan trọng lắm. Nếu lỡ `a < b` thì bước đầu `a % b = a`, kết quả là đổi chỗ `(b, a)` — tự sửa sau một vòng.

In [13]:
def gcd_iter(a, b):
    while b != 0:
        a, b = b, a % b
    return a

def find_gcd_euclid(nums):
    return gcd_iter(max(nums), min(nums))

print(find_gcd_euclid([2, 5, 6, 9, 10]))  # 2
print(find_gcd_euclid([7, 5, 6, 8, 3]))   # 1
print(find_gcd_euclid([3, 3]))            # 3
print("thứ tự ngược vẫn đúng:", gcd_iter(18, 48))

2
1
3
thứ tự ngược vẫn đúng: 6


## 7. TRACE từng bước

Hàm `trace_gcd` in trạng thái `(a, b)` **trước** mỗi lần cập nhật, phép chia và giá trị mới. Ta chạy trên:
- `[7, 5, 6, 8, 3]` → `gcd(8, 3)`: nguyên tố cùng nhau, đáp án 1.
- `[12, 30, 18, 42]` → `gcd(42, 12)`: có nhiều bước.

In [14]:
def trace_gcd(nums):
    mn, mx = min(nums), max(nums)
    print(f"nums = {nums} -> mn = {mn}, mx = {mx}")
    a, b = mx, mn
    step = 1
    while b != 0:
        q, r = divmod(a, b)          # divmod trả về (thương, dư)
        print(f"  Bước {step}: a = {a:>3}, b = {b:>3} | {a} = {q}·{b} + {r} -> (a, b) mới = ({b}, {r})")
        a, b = b, r
        step += 1
    print(f"  b = 0 -> GCD = {a}")
    return a

trace_gcd([7, 5, 6, 8, 3])
print()
trace_gcd([12, 30, 18, 42])

nums = [7, 5, 6, 8, 3] -> mn = 3, mx = 8
  Bước 1: a =   8, b =   3 | 8 = 2·3 + 2 -> (a, b) mới = (3, 2)
  Bước 2: a =   3, b =   2 | 3 = 1·2 + 1 -> (a, b) mới = (2, 1)
  Bước 3: a =   2, b =   1 | 2 = 2·1 + 0 -> (a, b) mới = (1, 0)
  b = 0 -> GCD = 1

nums = [12, 30, 18, 42] -> mn = 12, mx = 42
  Bước 1: a =  42, b =  12 | 42 = 3·12 + 6 -> (a, b) mới = (12, 6)
  Bước 2: a =  12, b =   6 | 12 = 2·6 + 0 -> (a, b) mới = (6, 0)
  b = 0 -> GCD = 6


6

### Bảng trace cho `[12, 30, 18, 42]` (mn = 12, mx = 42)

| Bước | a | b | Phép chia | (a, b) mới |
|:-:|:-:|:-:|---|:-:|
| 1 | 42 | 12 | 42 = 3·12 + 6 | (12, 6) |
| 2 | 12 | 6 | 12 = 2·6 + 0 | (6, 0) |
| — | 6 | 0 | dừng | GCD = **6** |

Ý nghĩa cột: `a`, `b` là cặp **trước** khi cập nhật; cột cuối là cặp sau phép gán đồng thời. Lưu ý `30` và `18` không ảnh hưởng gì: đề chỉ hỏi min và max.

Hàm `divmod(a, b)` mới xuất hiện: trả về tuple `(a // b, a % b)` trong một lần gọi.

In [15]:
print(divmod(42, 12))   # (3, 6)
print(divmod(12, 6))    # (2, 0)

(3, 6)
(2, 0)


## 8. Cách 3 — Euclid đệ quy

Công thức `gcd(a, b) = gcd(b, a % b)` viết thẳng thành hàm **đệ quy** (hàm tự gọi chính nó):

- **Điểm dừng (base case):** `b == 0` → trả `a`.
- **Bước đệ quy:** trả `gcd(b, a % b)`.

Độ sâu đệ quy chỉ khoảng `O(log V)` nên không lo tràn stack.

In [16]:
def gcd_rec(a, b):
    if b == 0:
        return a
    return gcd_rec(b, a % b)

def gcd_rec_trace(a, b, depth=0):
    print("  " * depth + f"gcd({a}, {b})")
    if b == 0:
        return a
    return gcd_rec_trace(b, a % b, depth + 1)

print("kết quả:", gcd_rec_trace(42, 12))
print("kết quả:", gcd_rec_trace(1000, 997))

gcd(42, 12)
  gcd(12, 6)
    gcd(6, 0)
kết quả: 6
gcd(1000, 997)
  gcd(997, 3)
    gcd(3, 1)
      gcd(1, 0)
kết quả: 1


### Code nộp LeetCode — Cách 3 (đệ quy)

In [17]:
class Solution:
    def findGCD(self, nums: List[int]) -> int:
        def gcd(a: int, b: int) -> int:
            return a if b == 0 else gcd(b, a % b)
        return gcd(max(nums), min(nums))

print(Solution().findGCD([7, 5, 6, 8, 3]))

1


## 9. Cách 4 — Dùng `math.gcd` (ngắn nhất)

Khi đã hiểu Euclid, trong thực tế ta dùng thư viện chuẩn: nhanh (viết bằng C), không lỗi.

Thêm một cách 5 để mở mang: **Euclid bằng phép trừ** — `gcd(a, b) = gcd(a - b, b)` khi `a > b`. Đúng, nhưng chậm hơn nhiều khi hai số chênh lệch lớn (ví dụ `gcd(1000, 1)` cần 999 lần trừ) — chính vì thế người ta dùng `%` (một phép `%` = gộp nhiều phép trừ).

In [18]:
def find_gcd_math(nums):
    return math.gcd(min(nums), max(nums))

def gcd_subtract(a, b):
    steps = 0
    while a != b:
        if a > b:
            a -= b
        else:
            b -= a
        steps += 1
    return a, steps

print(find_gcd_math([2, 5, 6, 9, 10]))
print("trừ dần (42, 12):", gcd_subtract(42, 12))
print("trừ dần (1000, 1):", gcd_subtract(1000, 1), "<- rất nhiều bước")

2
trừ dần (42, 12): (6, 4)
trừ dần (1000, 1): (1, 999) <- rất nhiều bước


## 10. Code nộp LeetCode (bản khuyến nghị)

Bản Euclid lặp: rõ ràng, không phụ thuộc thư viện, dễ trình bày khi phỏng vấn. Bản một dòng `math.gcd` cũng hoàn toàn được chấp nhận.

In [19]:
class Solution:
    def findGCD(self, nums: List[int]) -> int:
        a, b = max(nums), min(nums)
        while b:
            a, b = b, a % b
        return a


class SolutionMath:
    def findGCD(self, nums: List[int]) -> int:
        return math.gcd(min(nums), max(nums))


for nums, expected in [([2, 5, 6, 9, 10], 2), ([7, 5, 6, 8, 3], 1), ([3, 3], 3)]:
    assert Solution().findGCD(nums) == expected
    assert SolutionMath().findGCD(nums) == expected
print("Code nộp chạy đúng 3 ví dụ của đề")

Code nộp chạy đúng 3 ví dụ của đề


Lưu ý `while b:` tương đương `while b != 0:` vì số `0` được Python coi là `False`, số khác 0 là `True`.

In [20]:
for v in [0, 1, 6, -3]:
    print(v, "->", bool(v))

0 -> False
1 -> True
6 -> True
-3 -> True


## 11. Vì sao đúng? (invariant)

**Invariant của vòng `while`:** ở mọi thời điểm, `gcd(a, b) == gcd(mn, mx)`.

- **Khởi tạo:** `a = mx`, `b = mn` → đúng hiển nhiên.
- **Duy trì:** mỗi vòng thay `(a, b)` bằng `(b, a % b)`; ở mục 5 ta đã chứng minh hai cặp có **cùng tập ước chung** → GCD không đổi.
- **Kết thúc:** `b` giảm nghiêm ngặt (`0 ≤ a % b < b`) và là số nguyên không âm → chắc chắn chạm `0`. Khi `b = 0`: `gcd(a, 0) = a` (mọi số chia hết 0) → trả `a` là đúng.

Với Cách 1: ta thử `d` từ lớn xuống nhỏ trong đoạn `[1, mn]` — chứa **mọi** ước chung có thể — nên `d` đầu tiên thỏa là lớn nhất.

## 12. Time / Space và bảng so sánh

Gọi `n = len(nums)`, `V = max(nums)`.

Euclid: cứ **hai bước** thì số `a` giảm ít nhất một nửa (định lý Lamé), nên số vòng là `O(log V)`.

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Thử ước | `d` từ `mn` xuống 1 | `O(n + V)` | `O(1)` | Dễ nghĩ nhất, theo hint |
| 2. Euclid lặp | `(a,b) -> (b, a%b)` | `O(n + log V)` | `O(1)` | **Khuyến nghị** |
| 3. Euclid đệ quy | công thức đệ quy | `O(n + log V)` | `O(log V)` stack | Đẹp, ngắn |
| 4. `math.gcd` | thư viện | `O(n + log V)` | `O(1)` | Ngắn nhất |
| 5. Euclid trừ | `(a,b) -> (a-b, b)` | `O(n + V)` | `O(1)` | Chỉ để hiểu lịch sử |

Phần `O(n)` (tìm min/max) chiếm ưu thế ở mọi cách tốt; GCD chỉ thêm rất ít.

In [21]:
# Đếm số vòng Euclid để thấy O(log V)
def euclid_steps(a, b):
    steps = 0
    while b:
        a, b = b, a % b
        steps += 1
    return steps

for a, b in [(1000, 997), (987, 610), (10**18, 999999999989)]:
    print(f"gcd({a}, {b}) cần {euclid_steps(a, b)} vòng")

gcd(1000, 997) cần 3 vòng
gcd(987, 610) cần 14 vòng
gcd(1000000000000000000, 999999999989) cần 10 vòng


Cặp `(987, 610)` là hai số Fibonacci liên tiếp — trường hợp **tệ nhất** của Euclid — vẫn chỉ 14 vòng.

## 13. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

Các nhóm cần test:
1. Ba ví dụ của đề.
2. Biên: mảng 2 phần tử, `[1, 1000]` (gcd = 1), `[1000, 1000]`.
3. Phản ví dụ cho lỗi "GCD cả mảng": `[4, 6, 8]` → gcd(4, 8) = **4**, trong khi gcd cả mảng = 2.
4. min/max không nằm ở đầu/cuối mảng.
5. Đối chiếu ngẫu nhiên có seed giữa tất cả các cách.

In [22]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<32} PASS ({actual})")

check("đề 1", find_gcd_euclid([2, 5, 6, 9, 10]), 2)
check("đề 2", find_gcd_euclid([7, 5, 6, 8, 3]), 1)
check("đề 3", find_gcd_euclid([3, 3]), 3)
check("biên [1,1000]", find_gcd_euclid([1, 1000]), 1)
check("biên [1000,1000]", find_gcd_euclid([1000, 1000]), 1000)
check("phản ví dụ gcd cả mảng", find_gcd_euclid([4, 6, 8]), 4)
check("min/max ở giữa", find_gcd_euclid([50, 25, 100, 75]), 25)
check("nguyên tố lớn", find_gcd_euclid([997, 1000]), 1)

đề 1                             PASS (2)
đề 2                             PASS (1)
đề 3                             PASS (3)
biên [1,1000]                    PASS (1)
biên [1000,1000]                 PASS (1000)
phản ví dụ gcd cả mảng           PASS (4)
min/max ở giữa                   PASS (25)
nguyên tố lớn                    PASS (1)


In [23]:
import random

rng = random.Random(1979)
methods = [find_gcd_try_divisors, find_gcd_euclid, find_gcd_math,
           lambda ns: gcd_rec(max(ns), min(ns)), lambda ns: Solution().findGCD(ns)]
for _ in range(2000):
    nums = [rng.randint(1, 1000) for _ in range(rng.randint(2, 30))]
    answers = {m(nums) for m in methods}
    assert len(answers) == 1, (nums, answers)
print("2000 test ngẫu nhiên: mọi cách cho cùng đáp án")

2000 test ngẫu nhiên: mọi cách cho cùng đáp án


Giải thích `lambda`: `lambda ns: gcd_rec(max(ns), min(ns))` là một **hàm vô danh** viết trên một dòng, nhận `ns` và trả về biểu thức sau dấu `:`. Ta dùng để gói các cách có chữ ký khác nhau về cùng dạng `f(nums)`.

## 14. Lỗi người mới hay gặp

### Lỗi 1: Tính GCD của **cả mảng**

In [24]:
from functools import reduce

def wrong_gcd_all(nums):
    return reduce(math.gcd, nums)     # gcd(gcd(gcd(a0,a1),a2),...)

print("sai :", wrong_gcd_all([4, 6, 8]))      # 2
print("đúng:", find_gcd_euclid([4, 6, 8]))    # 4

sai : 2
đúng: 4


**Vì sao sai:** đề chỉ hỏi GCD của **min và max**. Phần tử ở giữa (6) kéo kết quả xuống. **Sửa:** chỉ lấy `min(nums)` và `max(nums)`.

(`functools.reduce(f, list)` gộp list bằng cách áp `f` lần lượt từ trái sang phải — đây là chủ đề của biến thể ở mục 16.)

### Lỗi 2: Gán tách dòng trong Euclid

In [25]:
def wrong_gcd_separate(a, b):
    while b:
        a = b
        b = a % b      # a đã bị ghi đè -> luôn ra 0
    return a

print("sai :", wrong_gcd_separate(42, 12))  # 12
print("đúng:", gcd_iter(42, 12))            # 6

sai : 12
đúng: 6


**Vì sao sai:** sau `a = b`, biểu thức `a % b` thành `b % b = 0`, vòng dừng ngay với `a = b` cũ. **Sửa:** dùng gán đồng thời `a, b = b, a % b`, hoặc biến tạm `r = a % b; a = b; b = r`.

### Lỗi 3: Thử ước đi **lên** và trả ngay ước đầu tiên

In [26]:
def wrong_try_up(nums):
    mn, mx = min(nums), max(nums)
    for d in range(1, mn + 1):
        if mn % d == 0 and mx % d == 0:
            return d          # luôn trả 1!
    return 1

print("sai :", wrong_try_up([2, 5, 6, 9, 10]))   # 1
print("đúng:", find_gcd_try_divisors([2, 5, 6, 9, 10]))  # 2

sai : 1
đúng: 2


**Vì sao sai:** đi từ 1 lên thì ước chung đầu tiên luôn là `1` (ước chung **nhỏ nhất**). **Sửa:** đi lùi từ `mn`, hoặc đi lên nhưng **ghi nhớ** ước chung cuối cùng thay vì `return` sớm.

### Lỗi 4: Khởi tạo `mn = 0` khi tự tìm min

In [27]:
def wrong_min(nums):
    mn = 0
    for x in nums:
        if x < mn:
            mn = x
    return mn

print("sai :", wrong_min([7, 5, 6, 8, 3]))   # 0, không phải 3
print("đúng:", min_max_one_pass([7, 5, 6, 8, 3])[0])

sai : 0
đúng: 3


**Vì sao sai:** mọi phần tử đều ≥ 1 > 0 nên không bao giờ cập nhật. **Sửa:** khởi tạo bằng `nums[0]` (hoặc `float('inf')`).

## 15. Mẹo nhận dạng pattern

- Thấy chữ **"greatest common divisor"**, **"chia đều"**, **"phân số tối giản"**, **"chu kỳ lặp chung"** → nghĩ tới **Euclid / `math.gcd`**.
- Thấy **"bội chung nhỏ nhất (LCM)"** → `lcm(a, b) = a // gcd(a, b) * b`.
- Đề có 2 bước (lọc dữ liệu rồi tính toán) → làm từng bước tách bạch, đừng gộp vội.
- Mẹo nhớ Euclid: **"chia lấy dư, đẩy sang trái, đến khi dư bằng 0"**.

## 16. Biến thể

1. **GCD của cả mảng** (ví dụ LeetCode 1071 dùng ý tương tự cho chuỗi): `reduce(math.gcd, nums)`.
2. **LCM của min và max**.
3. **Đếm số ước chung** của hai số: chính là số ước của `gcd(a, b)`.
4. **Rút gọn phân số** `p/q`: chia cả hai cho `gcd(p, q)`.

In [28]:
def lcm(a, b):
    return a // gcd_iter(a, b) * b     # chia trước nhân sau để số trung gian nhỏ

print("gcd cả mảng [12, 18, 30]:", reduce(gcd_iter, [12, 18, 30]))
print("lcm(4, 6):", lcm(4, 6))
g = gcd_iter(12, 18)
print("số ước chung của 12 và 18 = số ước của", g, "=", len(divisors(g)))
print("rút gọn 42/56:", 42 // gcd_iter(42, 56), "/", 56 // gcd_iter(42, 56))

gcd cả mảng [12, 18, 30]: 6
lcm(4, 6): 12
số ước chung của 12 và 18 = số ước của 6 = 4
rút gọn 42/56: 3 / 4


## 17. Bài tập tự luyện (TODO — chưa chạy)

### Bài 1 — Tính tay
Lập bảng Euclid cho `gcd(252, 105)`: mỗi dòng ghi `a`, `b`, `a % b`. Có bao nhiêu vòng? Đáp án là gì?

### Bài 2 — Dự đoán output
Không chạy code, đoán kết quả `find_gcd_euclid([9, 27, 6, 15])` và giải thích vì sao số 27 vẫn quan trọng còn 15 thì không.

### Bài 3 — Tự viết Euclid

In [29]:
def exercise_gcd(a: int, b: int) -> int:
    # TODO: dùng while và gán đồng thời a, b = b, a % b
    pass

# assert exercise_gcd(48, 18) == 6
# assert exercise_gcd(7, 13) == 1

### Bài 4 — Tự viết lời giải đầy đủ trong **một** lần duyệt mảng (không gọi `min`/`max`)

In [30]:
def exercise_find_gcd_one_pass(nums: list[int]) -> int:
    # TODO 1: khởi tạo mn, mx bằng nums[0]
    # TODO 2: duyệt và cập nhật mn, mx
    # TODO 3: Euclid trên (mx, mn)
    pass

# assert exercise_find_gcd_one_pass([2, 5, 6, 9, 10]) == 2
# assert exercise_find_gcd_one_pass([7, 5, 6, 8, 3]) == 1

### Bài 5 — Biến thể: LCM của số nhỏ nhất và lớn nhất

In [31]:
def exercise_lcm_min_max(nums: list[int]) -> int:
    # TODO: dùng công thức lcm = a // gcd(a, b) * b
    pass

# assert exercise_lcm_min_max([4, 5, 6]) == 12
# assert exercise_lcm_min_max([3, 3]) == 3

### Bài 6 — Mini challenge: đếm số cặp `(i, j)`, `i < j`, sao cho `gcd(nums[i], nums[j]) == 1`

In [32]:
def exercise_count_coprime_pairs(nums: list[int]) -> int:
    # TODO: hai vòng lặp i < j, dùng gcd
    pass

# assert exercise_count_coprime_pairs([2, 3, 4]) == 2   # (2,3), (3,4)
# assert exercise_count_coprime_pairs([2, 4, 6]) == 0

## 18. Gợi ý 3 mức, đáp án và tổng kết

### Gợi ý
| Bài | Mức 1 (hướng nghĩ) | Mức 2 (cấu trúc) | Mức 3 (gần code) |
|---|---|---|---|
| 1 | Chia lấy dư liên tục | Dừng khi dư = 0 | 252 = 2·105 + 42 … |
| 2 | Chỉ min và max quan trọng | min = 6, max = 27 | gcd(27, 6) |
| 3 | Công thức Euclid | `while b:` | `a, b = b, a % b; return a` |
| 4 | Một vòng for | `if x < mn: ...` | sau vòng: Euclid |
| 5 | lcm qua gcd | tìm min, max trước | `mn // g * mx` |
| 6 | Thử mọi cặp | `for i ... for j in range(i+1, n)` | `if gcd(...) == 1: cnt += 1` |

### Đáp án tham khảo (chạy kiểm tra)

- Bài 1: 252 = 2·105 + 42 → 105 = 2·42 + 21 → 42 = 2·21 + 0 → **3 vòng**, GCD = **21**.
- Bài 2: min = 6, max = 27 → gcd(27, 6) = **3**. Số 27 là max nên quyết định; 15 nằm giữa nên bị bỏ qua.

In [33]:
def answer_gcd(a, b):
    while b:
        a, b = b, a % b
    return a

def answer_find_gcd_one_pass(nums):
    mn = mx = nums[0]
    for x in nums:
        if x < mn:
            mn = x
        if x > mx:
            mx = x
    return answer_gcd(mx, mn)

def answer_lcm_min_max(nums):
    mn, mx = min(nums), max(nums)
    return mn // answer_gcd(mx, mn) * mx

def answer_count_coprime_pairs(nums):
    cnt = 0
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if answer_gcd(nums[i], nums[j]) == 1:
                cnt += 1
    return cnt

assert answer_gcd(252, 105) == 21
assert find_gcd_euclid([9, 27, 6, 15]) == 3
assert answer_gcd(48, 18) == 6 and answer_gcd(7, 13) == 1
assert answer_find_gcd_one_pass([2, 5, 6, 9, 10]) == 2
assert answer_find_gcd_one_pass([7, 5, 6, 8, 3]) == 1
assert answer_lcm_min_max([4, 5, 6]) == 12 and answer_lcm_min_max([3, 3]) == 3
assert answer_count_coprime_pairs([2, 3, 4]) == 2
assert answer_count_coprime_pairs([2, 4, 6]) == 0
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


### Template nhớ nhanh

```python
class Solution:
    def findGCD(self, nums: List[int]) -> int:
        a, b = max(nums), min(nums)
        while b:
            a, b = b, a % b      # chia lấy dư, đẩy sang trái
        return a
```

Một dòng: `return math.gcd(min(nums), max(nums))`.

### Checklist trước khi submit
- [ ] Mình lấy GCD của **min và max**, không phải cả mảng?
- [ ] Gán **đồng thời** `a, b = b, a % b` (không tách dòng)?
- [ ] Vòng `while` dừng khi `b == 0` và trả về `a` (không phải `b`)?
- [ ] Nếu thử ước: đi **lùi** từ `mn`, hoặc lưu ước cuối cùng?
- [ ] Đã test `[3, 3]` (min = max) và cặp nguyên tố cùng nhau (đáp án 1)?